# SimCell ODE Model Documentation

This document presents the kinetic ODE model for the synthetic E. coli SimCell from the notebook `SimCell_sim.ipynb`.

## Metabolic Map

The model simulates the following metabolic pathways:

- Lactate transport: $lac_e \xrightarrow[lldP/\mathrm{glycolate\_permease}]{v_{lacT}} lac_c$
- Lactate metabolism: $lac_c \xrightarrow[lldD + PDH]{v_1} AcCoA$
- TCA + ETC: $AcCoA \xrightarrow[v_{tca}]{TCA + ETC} ATP$
- Arginine biosynthesis: $AcCoA \xrightarrow[ArgD-H]{v_2} arg_c$
- Arginine export: $arg_c \xrightarrow[ArgO]{v_{argT}} arg_e$
- ATP maintenance and synthesis via PPK

## State Vector

The state vector $\mathbf{y}_1$ has 17 elements:

| Index | Variable | Description | Units |
|-------|----------|-------------|-------|
| 0 | $lac_c$ | Intracellular L-lactate | mM |
| 1 | $nh4_c$ | Intracellular ammonium | mM |
| 2 | $accoa$ | Acetyl-CoA | mM |
| 3 | $arg_c$ | Intracellular L-arginine | mM |
| 4 | $atp$ | ATP | mM |
| 5 | $adp$ | ADP | mM |
| 6 | $polyp$ | Polyphosphate | mM |
| 7 | $lac_e$ | Extracellular lactate | mM |
| 8 | $nh4_e$ | Extracellular ammonium | mM |
| 9 | $arg_e$ | Extracellular arginine | mM |
| 10 | $P_{lacT}$ | lldP + glycolate permease | mg/L |
| 11 | $P_{P1}$ | lldD + PDH | mg/L |
| 12 | $P_{tca}$ | TCA enzymes | mg/L |
| 13 | $P_{P2}$ | ArgD-H enzymes | mg/L |
| 14 | $P_{argT}$ | ArgO exporter | mg/L |
| 15 | $P_{ppk}$ | PPK | mg/L |
| 16 | $n_{atp}^{eff}$ | Effective ETC ATP yield | - |

## ODE System

The system of ODEs is:

$$\frac{d lac_c}{dt} = v_{lacT} - v_1$$

$$\frac{d nh4_c}{dt} = v_{nh3} + v_{nh4T} - v_2$$

$$\frac{d accoa}{dt} = v_1 - v_2 - v_{tca}$$

$$\frac{d arg_c}{dt} = v_2 - v_{argT}$$

$$\frac{d polyp}{dt} = -v_{ppk}$$

$$\frac{d atp}{dt} = n_{atp}^{eff} \cdot v_{tca} + v_{ppk} - 3 \cdot v_2 - \mathrm{atp\_repair} - v_{maint} - 0.1 \cdot v_{argT}$$

$$\frac{d adp}{dt} = -\frac{d atp}{dt}$$

$$\frac{d lac_e}{dt} = -\frac{v_{lacT}}{V_{ratio}}$$

$$\frac{d nh4_e}{dt} = -\frac{v_{nh3} + v_{nh4T}}{V_{ratio}}$$

$$\frac{d arg_e}{dt} = \frac{v_{argT}}{V_{ratio}}$$

Protein dynamics (for each protein class $P_i$):

$$\frac{d P_i}{dt} = v_r - D \cdot P_i$$

Where $D$ is the decay rate (0.003 hr⁻¹ for cytoplasmic/membrane proteins).

ETC capacity decay:

$$\frac{d n_{atp}^{eff}}{dt} = -D_{etc} \cdot n_{atp}^{eff}$$

## Reaction Rates

### Lactate transport (lldP + glycolate permease)
$$v_{lacT} = \frac{1}{2} \left( v_{raw} + \sqrt{v_{raw}^2 + \epsilon} \right)$$
where
$$v_{raw} = k_{lacT} \cdot P_{lacT} \cdot \left( \frac{lac_e}{K_{m,lacT,E} + lac_e} - \frac{lac_c}{K_{m,lacT,C} + lac_c} \cdot \frac{1}{K_{eq,lacT}} \right)$$

### NH4+/NH3 transport
$$v_{nh4T} = k_{nh4T} \cdot (nh4_e - nh4_c)$$
$$v_{nh3} = P_{nh3} \cdot (nh3_e \cdot nh4_e - nh3_c \cdot nh4_c)$$
where $nh3 = \frac{total_{nh4}}{1 + 10^{pKa - pH}}$

### lldD + PDH
$$v_1 = k_1 \cdot P_{P1} \cdot \frac{lac_c}{lac_c + K_{m1}}$$

### ArgD (rate-limiting arginine biosynthesis)
$$v_2 = k_{argD} \cdot P_{P2} \cdot \frac{accoa}{accoa + K_{m,argD}} \cdot \frac{nh4_c}{nh4_c + K_{m2,nh4}} \cdot \frac{atp}{atp + K_{m2,atp}}$$

### TCA + ETC
$$v_{tca} = k_{tca} \cdot P_{tca} \cdot \frac{accoa}{accoa + K_{m,tca}} \cdot \frac{adp}{adp + K_{m,adp,tca}}$$

### PPK
$$v_{ppk} = a_{ppk} \cdot P_{ppk} \cdot \frac{adp}{K_{m,adp} + adp} \cdot \frac{polyp}{K_{m,polyp} + polyp}$$

### Protein synthesis
$$v_r = r \cdot \frac{atp}{K_e + atp}$$
$$v_{r,total} = v_r \cdot N_{protein\_classes}$$
$$\mathrm{atp\_repair} = \mathrm{atp\_repair\_const} \cdot v_{r,total}$$
$$v_{maint} = m_{ATP} \cdot \phi$$

### ArgO export
$$v_{argT} = k_{argO} \cdot P_{argT} \cdot \frac{arg_c}{arg_c + K_{m,argO}}$$

## Parameters

All rates in mmol hr⁻¹ mg⁻¹ unless noted.

### Kinetic Parameters (K_PARAMS)

| Parameter | Value | Units | Description |
|-----------|-------|-------|-------------|
| $k_{lacT}$ | 0.00024 | mmol hr⁻¹ mg⁻¹ | Lactate transport rate |
| $K_{m,lacT,E}$ | 0.01 | mM | Km extracellular lactate |
| $K_{m,lacT,C}$ | 0.01 | mM | Km intracellular lactate |
| $K_{eq,lacT}$ | 189.0 | - | Equilibrium constant |
| $k_{nh4T}$ | 0.0001 | hr⁻¹ | NH4 ionic leak |
| $P_{nh3}$ | 3780000.0 | hr⁻¹ | NH3 diffusion permeability |
| $pKa$ | 9.25 | - | NH4+/NH3 equilibrium |
| $pH_{ext}$ | 7 | - | External pH |
| $pH_{int}$ | 7.5 | - | Internal pH |
| $k_1$ | 0.012 | mmol hr⁻¹ mg⁻¹ | lldD + PDH rate |
| $K_{m1}$ | 0.26 | mM | Km lactate |
| $k_{tca}$ | 0.0465 | mmol hr⁻¹ mg⁻¹ | TCA + ETC rate |
| $K_{m,tca}$ | 0.12 | mM | Km AcCoA |
| $K_{m,adp,tca}$ | 0.10 | mM | Km ADP |
| $k_{argD}$ | 3.84e-4 | mmol hr⁻¹ mg⁻¹ | ArgD rate |
| $K_{m,argD}$ | 0.48 | mM | Km AcCoA |
| $K_{m2,nh4}$ | 1.5 | mM | Km NH4 for ArgD |
| $K_{m2,atp}$ | 1.1 | mM | Km ATP for ArgD |
| $k_{argO}$ | 1.38e-4 | mmol hr⁻¹ mg⁻¹ | ArgO rate |
| $K_{m,argO}$ | 10.0 | mM | Km arginine |
| $a_{ppk}$ | 0.0642 | mmol hr⁻¹ mg⁻¹ | PPK rate |
| $K_{m,adp}$ | 0.25 | mM | Km ADP for PPK |
| $K_{m,polyp}$ | 0.15 | mM | Km polyP for PPK |
| $atp_{repair\_const}$ | 0.037 | mM ATP per mM protein | Repair cost |

### Decay Rates

| Parameter | Value | Units | Description |
|-----------|-------|-------|-------------|
| $D_{cyto}$ | 0.003 | hr⁻¹ | Cytoplasmic protein decay |
| $D_{mem}$ | 0.003 | hr⁻¹ | Membrane protein decay |
| $D_{etc}$ | 0.003 | hr⁻¹ | ETC complex decay |

### Protein Decay Map

| Protein | Decay Class |
|---------|-------------|
| $P_{lacT}$ | $D_{mem}$ |
| $P_{P1}$ | $D_{cyto}$ |
| $P_{tca}$ | $D_{cyto}$ |
| $P_{P2}$ | $D_{cyto}$ |
| $P_{argT}$ | $D_{mem}$ |
| $P_{ppk}$ | $D_{cyto}$ |

### Scaling and Initial Conditions

| Parameter | Value | Units | Description |
|-----------|-------|-------|-------------|
| $Cell\_density\_dcw$ | 45 | gDCW/L | Cell density |
| $V_{per\_dcw}$ | 2.0e-3 | L_cytoplasm/gDCW | Cytoplasm volume per DCW |
| $\phi_{cell}$ | 0.09 | - | Cell volume fraction |
| $V_{ratio}$ | 11.111 | L_bio/L_cell | Bioreactor to cell volume ratio |
| $PTOT\_to\_GDW$ | 550.0 | mg protein/g DW | Protein to dry weight |
| $P_{fracs}$ | Various | - | Initial protein fractions |

### Simulation Parameters

| Parameter | Value | Units | Description |
|-----------|-------|-------|-------------|
| $r$ | 10 | mg/L/hr | Protein synthesis rate |
| $K_e$ | 0.5 | mM | ATP half-sat for ribosomes |
| $m_{ATP}$ | 0.01 | mM/hr | Maintenance ATP |
| $Ptot0$ | 200000.0 | mg/L | Initial total protein |
| $atp0$ | 1.8 | mM | Initial ATP |
| $A_{tot}$ | 2.36 | mM | Total adenylate pool |
| $polyp0$ | 2.0 | mM | Initial polyphosphate |
| $nh4_{c0}$ | 0.1 | mM | Initial intracellular NH4 |
| $arg_{c0}$ | 0.01 | mM | Initial intracellular arginine |
| $n_{atp\_eff0}$ | 10.0 | - | Initial ETC yield |

## Parameter Values Summary
| Parameter | Value | Range | Source |
|---|---|---|---|
| k_lacT | 2.4e-5 mmol/hr*mg | Not explicitly given | 2.4e-5 mmol/hr*mg Transport of L-Lactate, D-Lactate, and glycolate by the LldP and GlcA membrane carriers of Escherichia coli Núñez MF, Kwon O, Wilson TH, Aguilar J, Baldoma L, Lin EC This value is against concentration gradient – not realistic for SimCell operation where internally lactate is consumed rapidly Lactate pooled uptake Estimating from LacY – the closest Major Facilitator Superfamily transporter and most studied, estimate 1 – 3 mmol hr-1 mg-1 This can be treated as a fitted parameter |
| Km_lacT_E | 0.01mM | Not explicitly given | Transport of L-Lactate, D-Lactate, and glycolate by the LldP and GlcA membrane carriers of Escherichia coli Núñez MF, Kwon O, Wilson TH, Aguilar J, Baldoma L, Lin EC |
| Km_lacT_C | 0.01mM | Not explicitly given | Transport of L-Lactate, D-Lactate, and glycolate by the LldP and GlcA membrane carriers of Escherichia coli Núñez MF, Kwon O, Wilson TH, Aguilar J, Baldoma L, Lin EC |
| Keq_lacT | Resting membrane potential− 140 mV as E. coli enters the late exponential phase. | Not explicitly given | Membrane potential: Corina Teodora Bot, Camelia Prodan, Quantifying the membrane potential during E. coli growth stages, Biophysical Chemistry, Volume 146, Issues 2–3, 2010, Pages 133-137, ISSN 0301-4622, https://doi.org/10.1016/j.bpc.2009.11.005. |
| K_nh4T | Negligible | Not explicitly given | Modelling nitrogen assimilation of Escherichia coli at low ammonium concentration Ma H, Boogerd FC, Goryanin I J. Biotechnol., 2009 |
| P_nh3 | 0.15 mm/s | Not explicitly given | Modelling nitrogen assimilation of Escherichia coli at low ammonium concentration Ma H, Boogerd FC, Goryanin I J. Biotechnol., 2009 |
| pKa | 9.25 | Not explicitly given | [Prokaryotic ammonium transporters: what has three decades of research revealed?] |
| Ph_ext | 7 | Not explicitly given | Modelling nitrogen assimilation of Escherichia coli at low ammonium concentration Ma H, Boogerd FC, Goryanin I J. Biotechnol., 2009 |
| pH_int | 7.5 | Not explicitly given | Modelling nitrogen assimilation of Escherichia coli at low ammonium concentration Ma H, Boogerd FC, Goryanin I J. Biotechnol., 2009 |
| k1 | 0.012 mmol/hr*mg | Not explicitly given | The pyruvate-dehydrogenase complex from Azotobacter vinelandii Bresters TW, de Abreu RA, de Kok A, Visser J, Veeger C Eur. J. Biochem., 1975 Kale S, Arjunan P, Furey W, Jordan F. 2007. A dynamic loop at the active center of the Escherichia coli pyruvate dehydrogenase complex E1 component modulates substrate utilization and chemical communication with the E2 component. J Biol Chem 282:28106–28116. |
| Km1 | 0.26 mM | Not explicitly given | Kale S, Arjunan P, Furey W, Jordan F. 2007. A dynamic loop at the active center of the Escherichia coli pyruvate dehydrogenase complex E1 component modulates substrate utilization and chemical communication with the E2 component. J Biol Chem 282:28106–28116. |
| k_tca | 0.0465 mmol/hrmg | Not explicitly given | Chakraborty J, Nemeria NS, Zhang X, et al. Engineering 2-oxoglutarate dehydrogenase to a 2-oxo aliphatic dehydrogenase complex by optimizing consecutive components. AIChE J. 2020;66:e16769. https://doi.org/10.1002/aic.16769 |
| Km_tca | 0.12 mM | Not explicitly given | Probing the roles of key residues in the unique regulatory NADH binding site of type II citrate synthase of Escherichia coli Stokell DJ, Donald LJ, Maurus R, Nguyen NT, Sadler G, Choudhary K et al. J. Biol. Chem., 2003 |
| Km_adp_tca | 0.5mM- 0.025mM | Not explicitly given | Nakamoto RK, Baylis Scanlon JA, Al-Shawi MK. The rotary mechanism of the ATP synthase. Arch Biochem Biophys. 2008 Aug 1;476(1):43-50. doi: 10.1016/j.abb.2008.05.004. Epub 2008 May 20. PMID: 18515057; PMCID: PMC2581510. |
| k_argD | 3.84e-4 | Not explicitly given | E.coli wild type [Billheimer, J.T.; Carnevale, H.N.; Leisinger, T.; Eckardt, T.; Jones, E.E. Ornithine delta-transaminase activity in Escherichia coli: its identity with acetylornithine delta-transaminase (1976), J. Bacteriol., 127, 1315-1323.] |
| Km_argD | 0.48mM | Not explicitly given | E.coli wild type [Billheimer, J.T.; Carnevale, H.N.; Leisinger, T.; Eckardt, T.; Jones, E.E. Ornithine delta-transaminase activity in Escherichia coli: its identity with acetylornithine delta-transaminase (1976), J. Bacteriol., 127, 1315-1323.] |
| Km2_nh4 | 3mM (for Chimeric) | Not explicitly given | Modular coenzyme specificity: a domain-swopped chimera of glutamate dehydrogenase Sharkey MA, Engel PC Proteins, 2009 |
| Km2_atp | 1.1mM | Not explicitly given | Two crystal structures of Escherichia coli N-acetyl-L-glutamate kinase demonstrate the cycling between open and closed conformations Gil-Ortiz F, Ramón-Maiques S, Fernández-Murga ML, Fita I, Rubio V J. Mol. Biol., 2010 |
| k_argO | 1.38e-4 | Not explicitly given | Estimated from arg production over time of SJB009 strain Ginesy, M., Belotserkovsky, J., Enman, J. et al. Metabolic engineering of Escherichia coli for enhanced arginine biosynthesis. Microb Cell Fact 14, 29 (2015). https://doi.org/10.1186/s12934-015-0211-y |
| Km_argO | 1.5mM -- 20mM | Not explicitly given | Estimated from LyseE exporter, could be lowered or fitted LyseE exporter for L-lysine and arginine Bröer S & Krämer R (1991) "Lysine excretion by Corynebacterium glutamicum. 1. Identification of a specific secretion carrier system." Eur J Biochem 202:131–135. Dubey S, Majumder P, Penmatsa A, Sardesai AA. Topological analyses of the L-lysine exporter LysO reveal a critical role for a conserved pair of intramembrane solvent-exposed acidic residues. J Biol Chem. 2021 Oct;297(4):101168. doi: 10.1016/j.jbc.2021.101168. Epub 2021 Sep 4. PMID: 34487760; PMCID: PMC8498466. |
| a_ppk | 0.0642 mmol/hr*mg | Not explicitly given | Metaphosphate synthesis by an enzyme from Escherichia coli Kornberg A, Kornberg SR, Simms ES Biochim. Biophys. Acta, 1956 |
| km_adp | 0.25 | Not explicitly given | Kuroda and Kornberg 1997 |
| km_polyp | 0.15mM | Not explicitly given | 0.2mM [Ahn K, Kornberg A. Polyphosphate kinase from Escherichia coli. Purification and demonstration of a phosphoenzyme intermediate. J Biol Chem. 1990 Jul 15;265(20):11734-9. PMID: 2164013.] 0.035mM [Tzeng C, Kornberg A The Multiple Activities of Polyphosphate Kinase ofEscherichia coli and Their Subunit Structure Determined by Radiation Target Analysis * Journal of Biological Chemistry, 2000; 275, 3977-3983] |
| atp_repair_const | N/A | Not explicitly given | Provided by Sizhe |
| D_cyto | 0.003 hr-1 | Not explicitly given | Chromosome free paper : half life of approximately 10 days observed in activity |
| D_mem | 0.003 hr-1 | Not explicitly given | Chromosome free paper : half life of approximately 10 days observed in activity |
| D_etc | 0.003hr-1 | Not explicitly given | Chromosome free paper : half life of approximately 10 days observed in activity |
| Ptot | 200 to 320 mg/ml | 200 to 320 | [Protein Mobility in the Cytoplasm of Escherichia coli Elowitz] |
| Ptot_to_gdw | 55% | Not explicitly given | Modelling nitrogen assimilation of Escherichia coli at low ammonium concentration Ma H, Boogerd FC, Goryanin I J. Biotechnol., 2009 |
| m_ATP | 1 mM/hr | Not explicitly given | Fitted 7.5 mmol/gDW/hr multiplied by the initial cell density of 0.36 gDW/L |
| r | N/A | Not explicitly given | Fitted |
| Ke | N/A | Not explicitly given | Fitted |
| Atp0 | 1–5 mM | 1 to 5 | 1-5mM range: [doi: 10.1186/1471-2180-13-301 Release of extracellular ATP by bacteria during growth] 9.6 mM Glucose-fed Bennett BD, Kimball EH, Gao M, Osterhout R, Van Dien SJ, Rabinowitz JD. Absolute metabolite concentrations and implied enzyme active site occupancy in Escherichia coli. Nat Chem Biol. 2009 Aug5(8):593-9 Conservative: Yaginuma et al., Diversity in ATP concentrations in a single bacterial cell population revealed by quantitative single-cell imaging. Sci Rep. 2014 Oct 6 4 :6522. doi: 10.1038/srep06522. p.4 right column bottom paragraph & p.5 table 1 & right column 2nd paragraph |
| Adp0 | 0.56mM (Glucose-fed) | Not explicitly given | Bennett BD, Kimball EH, Gao M, Osterhout R, Van Dien SJ, Rabinowitz JD. Absolute metabolite concentrations and implied enzyme active site occupancy in Escherichia coli. Nat Chem Biol. 2009 Aug5(8):593-9 |
| Lac_e and nh4_e set by Sizhe paper conditions (set to 0 in coupled model) | N/A | Not explicitly given | None provided |